# Synthetic student-risk classification (KNN)
A standalone BAT 405 model exercise. The dashboard does not use this model. All records and outcomes are fictional.


In [ ]:
%pip -q install scikit-learn==1.6.1


In [ ]:
import csv, json, math, random
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 405
FEATURES = ['previous_average_grade', 'attendance_percent', 'previous_failed_subjects', 'assignment_completion_percent']

def student(rng, key, with_outcome=False):
    # Fictional 0–100 grade scale, not an assertion about any university's grading.
    preparedness = rng.gauss(0, 1)
    clip = lambda value, low, high: max(low, min(high, value))
    grade = round(clip(77 + 10 * preparedness + rng.gauss(0, 6), 35, 99), 1)
    attendance = round(clip(84 + 9 * preparedness + rng.gauss(0, 8), 35, 100), 1)
    completion = round(clip(80 + 12 * preparedness + rng.gauss(0, 10), 20, 100), 1)
    failed = int(clip(round(0.7 - 0.65 * preparedness + rng.gauss(0, 0.8)), 0, 5))
    row = dict(zip(FEATURES, [grade, attendance, failed, completion]))
    row = {'record_key': key, **row}
    if with_outcome:
        # A probabilistic fictional next-semester outcome. No deterministic threshold label.
        log_odds = -1.9 + (75-grade)*0.065 + (80-attendance)*0.045 + failed*0.6 + (75-completion)*0.035
        probability = 1 / (1 + math.exp(-log_odds))
        row['failed_next_semester'] = int(rng.random() < probability)
    return row


def scores(actual, predicted):
    return { 'accuracy': accuracy_score(actual, predicted),
             'precision': precision_score(actual, predicted, zero_division=0),
             'recall': recall_score(actual, predicted, zero_division=0),
             'f1': f1_score(actual, predicted, zero_division=0),
             'confusion_matrix': confusion_matrix(actual, predicted, labels=[0,1]).tolist() }


def main():
    rng = random.Random(SEED)
    rows = [student(rng, f'training-demo-{i+1:04}', True) for i in range(1000)]
    train, test = train_test_split(rows, test_size=0.2, stratify=[r['failed_next_semester'] for r in rows], random_state=SEED)
    x = lambda records: [[r[f] for f in FEATURES] for r in records]
    y = lambda records: [r['failed_next_semester'] for r in records]
    pipeline = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier())])
    search = GridSearchCV(pipeline, {'knn__n_neighbors':[3,5,7,9,11,15,21,31], 'knn__weights':['uniform','distance']},
                          scoring='recall', cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=1)
    search.fit(x(train), y(train))
    predicted = search.predict(x(test))
    baseline = DummyClassifier(strategy='most_frequent').fit(x(train), y(train))
    metrics = scores(y(test), predicted)
    test_keys = {r['record_key'] for r in test}
    assert not test_keys.intersection(r['record_key'] for r in train)
    assert sum(sum(row) for row in metrics['confusion_matrix']) == len(test)
    results = {
        'data_status':'Synthetic demonstration only; no real student records or validated institutional predictions.',
        'target':'Whether a fictional student fails at least one subject in the following semester.',
        'seed':SEED, 'sklearn_version':sklearn.__version__, 'features':FEATURES,
        'training_records':len(train), 'test_records':len(test),
        'selection':'Five-fold stratified CV on training data; maximize recall. Scaler fitted separately inside each fold.',
        'best_parameters':search.best_params_, 'cv_recall':search.best_score_,
        'test_metrics':metrics, 'majority_baseline_test_metrics':scores(y(test),baseline.predict(x(test))),
        'confusion_matrix_order':'Rows actual [not at risk, at risk]; columns predicted [not at risk, at risk].',
    }
    print(f"Training records: {len(train)} | Test records: {len(test)}")
    print(f"Selected KNN: k={search.best_params_['knn__n_neighbors']}, weights={search.best_params_['knn__weights']}")
    print(f"Cross-validation recall: {search.best_score_:.1%}")
    for name in ('accuracy', 'precision', 'recall', 'f1'):
        print(f"Test {name}: {metrics[name]:.1%}")
    print("Confusion matrix [actual rows 0/1, predicted columns 0/1]:")
    print(metrics['confusion_matrix'])
    print("Synthetic demonstration only; not validated for real student decisions.")



main()
